# KDTree Data Structure Example

This notebook demonstrates how to use the `KDTree` data structure provided by `geocutool` for efficient nearest neighbor queries on a point cloud. We'll start by importing the necessary libraries.

In [1]:
from geocutool.data_structure import KDTree

import torch
import numpy as np
import plotly.graph_objects as go
import open3d as o3d
import time

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## 1. Generate a Point Cloud

We generate a 3D point cloud by sampling the vertices of a sphere mesh created using Open3D. The resulting numpy array is converted to a PyTorch tensor for processing.

In [2]:
sphere_mesh = o3d.geometry.TriangleMesh.create_sphere(radius=1.0, resolution=20)
points = np.asarray(sphere_mesh.vertices, dtype=np.float32)
points_tensor = torch.from_numpy(points)

print(points_tensor.shape)

torch.Size([762, 3])


## 2. Build KDTree and Query

Next, we transfer our point cloud tensor to the GPU, build a `KDTree` using `geocutool`, and query the 25 nearest neighbors for a specific point in the cloud. We use `exclude_self=True` to exclude the query point itself from the results.

In [3]:
points_tensor = points_tensor.to('cuda')

kdtree = KDTree(points_tensor)

query_index = 250
query_point = points_tensor[query_index].unsqueeze(0)
k = 25

start_time = time.time()
dists, indices = kdtree.query(query_point, k, exclude_self=True)
end_time = time.time()

print(f"Query Time: {end_time - start_time:.6f} seconds")

print("Query Point:", query_point)
print("Distances:", dists)
print("Indices:", indices)

Query Time: 0.000073 seconds
Query Point: tensor([[0.2753, 0.8474, 0.4540]], device='cuda:0')
Distances: tensor([[0.0195, 0.0195, 0.0246, 0.0246, 0.0424, 0.0424, 0.0455, 0.0455, 0.0777,
         0.0777, 0.0952, 0.0952, 0.0979, 0.0979, 0.1076, 0.1076, 0.1134, 0.1134,
         0.1196, 0.1196, 0.1596, 0.1596, 0.1731, 0.1731, 0.1818]],
       device='cuda:0')
Indices: tensor([[251, 249, 290, 210, 209, 211, 291, 289, 252, 248, 212, 208, 170, 330,
         292, 288, 171, 169, 331, 329, 168, 172, 247, 253, 213]],
       device='cuda:0')


## 3. Visualize the Results

Finally, we visualize the query point, its neighbors, and the rest of the point cloud using Plotly. Neighbors are shown in red, the query point in green, and the non-neighbors in blue.

In [4]:
query_point_np = query_point.cpu().numpy()[0]
neighbor_points_np = points_tensor[indices[0]].cpu().numpy()
mask = torch.ones(points_tensor.size(0), dtype=torch.bool, device=points_tensor.device)
mask[indices[0]] = False
non_neighbor_points_np = points_tensor[mask].cpu().numpy()

print("Neighbor Points:\n", neighbor_points_np.shape)
print("Non-Neighbor Points:\n", non_neighbor_points_np.shape)

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=non_neighbor_points_np[:, 0], 
        y=non_neighbor_points_np[:, 1], 
        z=non_neighbor_points_np[:, 2], 
        mode='markers', 
        marker=dict(size=5, color='blue'), 
        name='Non-Neighbors'))
fig.add_trace(
    go.Scatter3d(
        x=neighbor_points_np[:, 0], 
        y=neighbor_points_np[:, 1], 
        z=neighbor_points_np[:, 2], 
        mode='markers', 
        marker=dict(size=5, color='red'), 
        name='Neighbors'))
fig.add_trace(
    go.Scatter3d(
        x=[query_point_np[0]], 
        y=[query_point_np[1]], 
        z=[query_point_np[2]], 
        mode='markers', 
        marker=dict(size=10, color='green'), 
        name='Query Point'))

fig.update_layout(
    title='KD-Tree Nearest Neighbors', 
    scene=dict(
        xaxis_title='X', 
        yaxis_title='Y', 
        zaxis_title='Z'
    )
)

fig.show()

Neighbor Points:
 (25, 3)
Non-Neighbor Points:
 (737, 3)
